In [1]:
!pip install seaborn
!pip install xgboost
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np 
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
data=pd.DataFrame()
df_tabular=pd.read_csv("data/train/train_tabular.csv")
df_tabular.head()
df_tabular.info()


<class 'pandas.DataFrame'>
RangeIndex: 15872 entries, 0 to 15871
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   flight_id                 15872 non-null  int64  
 1   drone_id                  15872 non-null  str    
 2   model                     15872 non-null  str    
 3   motor_type                15872 non-null  str    
 4   firmware_version          15872 non-null  str    
 5   battery_capacity_mAh      15872 non-null  int64  
 6   max_payload_g             15872 non-null  float64
 7   propeller_in              15872 non-null  float64
 8   manufacture_batch         15872 non-null  int64  
 9   operator_region           15872 non-null  str    
 10  flight_index              15872 non-null  int64  
 11  payload_g                 15872 non-null  float64
 12  ambient_temp_C            15872 non-null  float64
 13  wind_speed_ms             15872 non-null  float64
 14  flight_duration_m

In [3]:
data=pd.DataFrame()
df_tabular_test=pd.read_csv("data/test/test_tabular.csv")
df_tabular_test.head()
df_tabular_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 7534 entries, 0 to 7533
Data columns (total 19 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   flight_id                 7534 non-null   int64  
 1   drone_id                  7534 non-null   str    
 2   model                     7534 non-null   str    
 3   motor_type                7534 non-null   str    
 4   firmware_version          7534 non-null   str    
 5   battery_capacity_mAh      7534 non-null   int64  
 6   max_payload_g             7534 non-null   float64
 7   propeller_in              7534 non-null   float64
 8   manufacture_batch         7534 non-null   int64  
 9   operator_region           7534 non-null   str    
 10  flight_index              7534 non-null   int64  
 11  payload_g                 7534 non-null   float64
 12  ambient_temp_C            7534 non-null   float64
 13  wind_speed_ms             7534 non-null   float64
 14  flight_duration_min

In [4]:
print(df_tabular["model"].value_counts())
print(df_tabular_test["model"].value_counts())

model
A    4624
C    4116
D    3609
B    3523
Name: count, dtype: int64
model
E    2336
A    1627
D    1219
B    1192
C    1160
Name: count, dtype: int64


In [5]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

In [6]:
# Cell 4 - Setup encoders
from sklearn.preprocessing import OneHotEncoder, TargetEncoder

# Target encoding for 'model' (has unseen category E in test)
te = TargetEncoder(target_type="binary", smooth="auto", random_state=42)

# OHE for the rest (no unseen categories)
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

ohe_cols = ["motor_type", "firmware_version", "operator_region"]

In [7]:
from sklearn.model_selection import GroupKFold
from sklearn.base import clone
import pandas as pd
import numpy as np

X_train_tabular = df_tabular.drop(columns=["drone_id", "flight_id", "failure_within_horizon", "failure_mode"])
labels = df_tabular["failure_within_horizon"]


categorical_all = ["model", "motor_type", "firmware_version", "operator_region"]
numerical_cols = [c for c in X_train_tabular.columns if c not in categorical_all]
ohe_cols = ["motor_type", "firmware_version", "operator_region"]


X_num = X_train_tabular[numerical_cols].copy()
X_num["payload_ratio"] = X_num["payload_g"] / X_num["max_payload_g"]
X_num["throttle_x_wind"] = X_num["avg_throttle"] * X_num["wind_speed_ms"]
X_num["maneuvers_per_min"] = X_num["num_aggressive_maneuvers"] / X_num["flight_duration_min"]
X_num["cycles_per_hour"] = X_num["battery_cycles"] / (X_num["cumulative_flight_hours"] + 0.01)
X_num["battery_cycles_sq"] = X_num["battery_cycles"] ** 2

ohe_enc_train = ohe.fit_transform(df_tabular[ohe_cols])
ohe_train_dense = ohe_enc_train.toarray() if hasattr(ohe_enc_train, 'toarray') else ohe_enc_train
ohe_names = ohe.get_feature_names_out(ohe_cols)
ohe_enc_train_df = pd.DataFrame(ohe_train_dense, columns=ohe_names, index=X_train_tabular.index)

X_train_base = pd.concat([X_num, ohe_enc_train_df], axis=1)


oof_model_enc = np.zeros(len(df_tabular))
gkf = GroupKFold(n_splits=6)
groups = df_tabular["drone_id"]

for train_idx, val_idx in gkf.split(df_tabular, labels, groups=groups):
    
    X_tr, y_tr = df_tabular.iloc[train_idx], labels.iloc[train_idx]
    X_val = df_tabular.iloc[val_idx]
    
    
    fold_te = clone(te)
    fold_te.fit(X_tr[["model"]], y_tr)
    
    
    val_enc = fold_te.transform(X_val[["model"]])
    oof_model_enc[val_idx] = val_enc.values.flatten() if hasattr(val_enc, 'columns') else val_enc.flatten()


X_encoded_tabular = X_train_base.copy()
X_encoded_tabular["model_enc"] = oof_model_enc
X_encoded_tabular.drop(columns=[
    'battery_cycles',          # battery_cycles_sq captures this better
    'wind_speed_ms',           # throttle_x_wind captures this
    'avg_throttle',            # throttle_x_wind captures this
    'payload_g',               # payload_ratio captures this
]
)
print("Train shape:", X_encoded_tabular.shape)

Train shape: (15872, 29)


In [8]:
te.fit(df_tabular[["model"]], labels)

X_test_num = df_tabular_test[numerical_cols].copy()
X_test_num["payload_ratio"] = X_test_num["payload_g"] / X_test_num["max_payload_g"]
X_test_num["throttle_x_wind"] = X_test_num["avg_throttle"] * X_test_num["wind_speed_ms"]
X_test_num["maneuvers_per_min"] = X_test_num["num_aggressive_maneuvers"] / X_test_num["flight_duration_min"]
X_test_num["cycles_per_hour"] = X_test_num["battery_cycles"] / (X_test_num["cumulative_flight_hours"] + 0.01)
X_test_num["battery_cycles_sq"] = X_test_num["battery_cycles"] ** 2


model_enc_test = te.transform(df_tabular_test[["model"]])
model_test_vals = model_enc_test.values if hasattr(model_enc_test, 'columns') else model_enc_test
model_enc_test_df = pd.DataFrame(model_test_vals, columns=["model_enc"], index=df_tabular_test.index)

ohe_enc_test = ohe.transform(df_tabular_test[ohe_cols])
ohe_test_dense = ohe_enc_test.toarray() if hasattr(ohe_enc_test, 'toarray') else ohe_enc_test
ohe_enc_test_df = pd.DataFrame(ohe_test_dense, columns=ohe_names, index=df_tabular_test.index)

X_encoded_tabular_test = pd.concat([X_test_num, model_enc_test_df, ohe_enc_test_df], axis=1)
print("Test shape:", X_encoded_tabular_test.shape)
X_encoded_tabular_test = X_encoded_tabular_test[X_encoded_tabular.columns]
X_encoded_tabular_test.drop(columns=[
    'battery_cycles',          # battery_cycles_sq captures this better
    'wind_speed_ms',           # throttle_x_wind captures this
    'avg_throttle',            # throttle_x_wind captures this
    'payload_g',               # payload_ratio captures this
]
)

Test shape: (7534, 29)


,battery_capacity_mAh,max_payload_g,propeller_in,manufacture_batch,flight_index,ambient_temp_C,flight_duration_min,num_aggressive_maneuvers,cumulative_flight_hours,payload_ratio,...,motor_type_M2,motor_type_M3,firmware_version_v3.1,firmware_version_v3.2,firmware_version_v4.0,operator_region_east,operator_region_north,operator_region_south,operator_region_west,model_enc
0,5200,2200.0,11.0,8,1,31.99,17.30,3,24.79,0.074864,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.128299
1,5200,2200.0,11.0,8,2,20.53,7.47,7,24.92,0.786773,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.128299
2,5200,2200.0,11.0,8,3,17.45,15.75,5,25.18,0.716909,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.128299
3,5200,2200.0,11.0,8,4,34.90,20.34,1,25.52,0.342500,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.128299
4,5200,2200.0,11.0,8,5,28.85,17.91,5,25.82,0.999182,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.128299
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7529,8000,2200.0,15.0,0,11,10.33,17.13,6,30.14,0.808636,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.125126
7530,8000,2200.0,15.0,0,12,21.91,17.76,5,30.44,0.869000,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.125126
7531,8000,2200.0,15.0,0,13,23.79,8.17,0,30.57,0.229773,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.125126
7532,8000,2200.0,15.0,0,14,32.88,18.13,4,30.88,0.907000,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.125126


In [9]:
df_signal=pd.read_csv("data/train/flight_signals_dataset.csv")

df_signal = df_signal.merge(
    df_tabular[["flight_id", "failure_within_horizon","drone_id"]],
    on="flight_id",
    how="left"
)
df_signal.info()

<class 'pandas.DataFrame'>
RangeIndex: 2031616 entries, 0 to 2031615
Data columns (total 12 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   flight_id               int64  
 1   time_step               int64  
 2   accel_x                 float64
 3   accel_y                 float64
 4   accel_z                 float64
 5   gyro_z                  float64
 6   motor_current_1         float64
 7   motor_current_2         float64
 8   vibration               float64
 9   battery_voltage         float64
 10  failure_within_horizon  int64  
 11  drone_id                str    
dtypes: float64(8), int64(3), str(1)
memory usage: 186.0 MB


In [10]:

with np.load('data/test/test_signals.npz') as data:
    flight_ids = data['flight_id']
    signals = data['signals']
    
    label_key = [k for k in data.files if k not in ['flight_id', 'signals']][0]
    sensor_names = data[label_key]


n_flights, n_timesteps, n_sensors = signals.shape
flattened_signals = signals.reshape(-1, n_sensors)


df_flight_ids = np.repeat(flight_ids, n_timesteps)

df_time_steps = np.tile(np.arange(n_timesteps), n_flights)


df_signal_test = pd.DataFrame(flattened_signals, columns=sensor_names)
df_signal_test.insert(0, 'flight_id', df_flight_ids)
df_signal_test.insert(1, 'time_step', df_time_steps)


print("\nFirst few roxws:")
df_signal_test = df_signal_test.merge(
    df_tabular_test[["flight_id","drone_id"]],
    on="flight_id",
    how="left"
)
df_signal_test.head(200)




First few roxws:


,flight_id,time_step,accel_x,accel_y,accel_z,gyro_z,motor_current_1,motor_current_2,vibration,battery_voltage,drone_id
0,45,0,0.026406,-0.070378,-0.042789,0.023866,0.039605,-0.020819,0.017150,0.907727,D1002
1,45,1,-0.010014,-0.077875,-0.016147,0.052098,0.150264,-0.065170,0.077035,1.051151,D1002
2,45,2,0.012607,-0.042678,0.023671,0.014427,0.003560,0.066461,-0.049857,1.003242,D1002
3,45,3,0.055643,-0.032468,0.060638,0.054332,0.060988,0.014646,0.163096,1.018358,D1002
4,45,4,-0.049332,-0.080423,-0.016975,-0.052293,-0.011090,-0.008266,-0.058418,0.932074,D1002
...,...,...,...,...,...,...,...,...,...,...,...
195,46,67,0.203272,-0.068006,0.197075,0.120562,0.384624,0.349688,-0.096912,1.087523,D1002
196,46,68,0.173498,-0.066793,0.086413,-0.007592,0.437228,0.335254,-0.107740,0.999630,D1002
197,46,69,0.154501,-0.081355,0.077731,0.025907,0.486043,0.510195,0.057620,1.047918,D1002
198,46,70,0.205755,-0.133964,0.011303,0.005561,0.487346,0.431952,0.085252,1.062471,D1002


In [11]:
print(df_signal.info())
print(df_signal_test.info())

<class 'pandas.DataFrame'>
RangeIndex: 2031616 entries, 0 to 2031615
Data columns (total 12 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   flight_id               int64  
 1   time_step               int64  
 2   accel_x                 float64
 3   accel_y                 float64
 4   accel_z                 float64
 5   gyro_z                  float64
 6   motor_current_1         float64
 7   motor_current_2         float64
 8   vibration               float64
 9   battery_voltage         float64
 10  failure_within_horizon  int64  
 11  drone_id                str    
dtypes: float64(8), int64(3), str(1)
memory usage: 186.0 MB
None
<class 'pandas.DataFrame'>
RangeIndex: 964352 entries, 0 to 964351
Data columns (total 11 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   flight_id        964352 non-null  int64  
 1   time_step        964352 non-null  int64  
 2   accel_x          96435

In [12]:
sensor_cols = [
    "accel_x",
    "accel_y",
    "accel_z",
    "gyro_z",
    "motor_current_1",
    "motor_current_2",
    "vibration",
    "battery_voltage",
]

In [13]:
import numpy as np
from scipy.fft import rfft, rfftfreq

def fft_features(signal):

    fft = np.abs(rfft(signal))

    freqs = rfftfreq(len(signal), d=1)

    fft[0] = 0

    energy = np.sum(fft**2)

    peak_amp = np.max(fft)

    dominant_freq = freqs[np.argmax(fft)]

    spectral_centroid = np.sum(freqs * fft) / (np.sum(fft) + 1e-12)

    return {
        "fft_energy": energy,
        "fft_peak_amp": peak_amp,
        "fft_dominant_freq": dominant_freq,
        "fft_centroid": spectral_centroid,
    }

In [14]:
from scipy.stats import skew, kurtosis

all_features = []

for flight_id, flight in df_signal.groupby("flight_id"):

    features = {}

    for sensor in sensor_cols:

        signal = flight[sensor].values

        features[f"{sensor}_mean"] = np.mean(signal)

        features[f"{sensor}_std"] = np.std(signal)

        features[f"{sensor}_var"] = np.var(signal)

        features[f"{sensor}_rms"] = np.sqrt(np.mean(signal**2))

        features[f"{sensor}_min"] = np.min(signal)

        features[f"{sensor}_max"] = np.max(signal)

        features[f"{sensor}_median"] = np.median(signal)

        features[f"{sensor}_skew"] = skew(signal)

        features[f"{sensor}_kurtosis"] = kurtosis(signal)

        fft = np.abs(rfft(signal))

        freqs = rfftfreq(len(signal), d=1)

        features[f"{sensor}_fft_energy"] = np.sum(fft**2)

        features[f"{sensor}_fft_peak_amp"] = np.max(fft)

        features[f"{sensor}_fft_dominant_freq"] = freqs[np.argmax(fft)]

        features[f"{sensor}_fft_centroid"] = (
            np.sum(freqs * fft) /
            (np.sum(fft) + 1e-12)
        )

    features["flight_id"] = flight_id

    features["failure_within_horizon"] = flight["failure_within_horizon"].iloc[0]

    features["drone_id"] = flight["drone_id"].iloc[0]

    all_features.append(features)
signal_features = pd.DataFrame(all_features)
signal_features.head()

,accel_x_mean,accel_x_std,accel_x_var,accel_x_rms,accel_x_min,accel_x_max,accel_x_median,accel_x_skew,accel_x_kurtosis,accel_x_fft_energy,...,battery_voltage_median,battery_voltage_skew,battery_voltage_kurtosis,battery_voltage_fft_energy,battery_voltage_fft_peak_amp,battery_voltage_fft_dominant_freq,battery_voltage_fft_centroid,flight_id,failure_within_horizon,drone_id
0,-0.003113,0.064691,0.004185,0.064765,-0.148643,0.152975,-0.006182,0.144844,-0.411006,34.691350,...,0.993113,-0.025363,0.752568,16363.584168,127.827397,0.0,0.052868,0,0,D1000
1,0.120029,0.136075,0.018517,0.181448,-0.104294,0.694158,0.088490,2.124068,5.613133,387.775537,...,0.996783,0.367886,1.023691,16295.530827,127.569484,0.0,0.052931,1,0,D1000
2,0.053043,0.216955,0.047070,0.223345,-0.167692,0.783391,-0.003016,2.377044,4.620941,431.692818,...,0.998419,0.149473,0.207957,16443.910496,128.151964,0.0,0.051411,2,0,D1000
3,-0.051328,0.363990,0.132489,0.367591,-1.579541,1.088738,-0.091752,0.212041,5.315132,1128.729979,...,0.999793,-0.122443,-0.053000,16375.867039,127.885426,0.0,0.055403,3,0,D1000
4,-0.100770,0.181908,0.033091,0.207955,-0.915355,0.088882,-0.067879,-3.470190,12.159686,437.667077,...,1.003257,0.048934,-0.300196,16507.482280,128.372664,0.0,0.057884,4,0,D1000


In [15]:
from scipy.stats import skew, kurtosis

all_features = []

for flight_id, flight in df_signal_test.groupby("flight_id"):

    features = {}

    for sensor in sensor_cols:

        signal = flight[sensor].values

        features[f"{sensor}_mean"] = np.mean(signal)

        features[f"{sensor}_std"] = np.std(signal)

        features[f"{sensor}_var"] = np.var(signal)

        features[f"{sensor}_rms"] = np.sqrt(np.mean(signal**2))

        features[f"{sensor}_min"] = np.min(signal)

        features[f"{sensor}_max"] = np.max(signal)

        features[f"{sensor}_median"] = np.median(signal)

        features[f"{sensor}_skew"] = skew(signal)

        features[f"{sensor}_kurtosis"] = kurtosis(signal)

        fft = np.abs(rfft(signal))

        freqs = rfftfreq(len(signal), d=1)

        features[f"{sensor}_fft_energy"] = np.sum(fft**2)

        features[f"{sensor}_fft_peak_amp"] = np.max(fft)

        features[f"{sensor}_fft_dominant_freq"] = freqs[np.argmax(fft)]

        features[f"{sensor}_fft_centroid"] = (
            np.sum(freqs * fft) /
            (np.sum(fft) + 1e-12)
        )

    features["flight_id"] = flight_id

    features["drone_id"] = flight["drone_id"].iloc[0]

    all_features.append(features)
signal_features_test = pd.DataFrame(all_features)
signal_features_test.head()

,accel_x_mean,accel_x_std,accel_x_var,accel_x_rms,accel_x_min,accel_x_max,accel_x_median,accel_x_skew,accel_x_kurtosis,accel_x_fft_energy,...,battery_voltage_max,battery_voltage_median,battery_voltage_skew,battery_voltage_kurtosis,battery_voltage_fft_energy,battery_voltage_fft_peak_amp,battery_voltage_fft_dominant_freq,battery_voltage_fft_centroid,flight_id,drone_id
0,-0.030596,0.392554,0.154099,0.393745,-1.118119,1.474698,-0.063203,1.499813,7.539363,1277.801636,...,1.161372,0.999340,0.454964,0.104867,16493.474609,128.336975,0.0,0.052570,45,D1002
1,0.174113,0.356822,0.127322,0.397036,-0.345021,1.198304,0.040781,1.564517,1.405070,1539.712036,...,1.171959,1.000862,0.303643,0.181290,16452.894531,128.180893,0.0,0.054667,46,D1002
2,0.170454,0.626482,0.392480,0.649257,-0.220516,3.630040,-0.042273,2.901132,8.633787,3691.936523,...,1.152328,0.996529,0.153398,0.198983,16282.570312,127.519264,0.0,0.052105,47,D1002
3,-0.106690,0.064162,0.004117,0.124497,-0.263902,0.073125,-0.103008,0.082149,-0.120688,220.676605,...,1.169531,1.009433,-0.006514,0.659427,16603.175781,128.760651,0.0,0.054696,48,D1002
4,0.115025,0.312897,0.097905,0.333370,-0.447915,1.268018,0.039922,2.271366,4.845612,1018.826233,...,1.115806,1.000086,-0.280176,-0.288927,16344.332031,127.733025,0.0,0.054356,49,D1002


In [16]:

signal_features_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 7534 entries, 0 to 7533
Columns: 106 entries, accel_x_mean to drone_id
dtypes: float32(88), float64(16), int64(1), str(1)
memory usage: 3.6 MB


In [17]:
X_train_signal=signal_features.drop(columns=["flight_id","failure_within_horizon","drone_id"])
print(X_train_signal.columns.tolist())


['accel_x_mean', 'accel_x_std', 'accel_x_var', 'accel_x_rms', 'accel_x_min', 'accel_x_max', 'accel_x_median', 'accel_x_skew', 'accel_x_kurtosis', 'accel_x_fft_energy', 'accel_x_fft_peak_amp', 'accel_x_fft_dominant_freq', 'accel_x_fft_centroid', 'accel_y_mean', 'accel_y_std', 'accel_y_var', 'accel_y_rms', 'accel_y_min', 'accel_y_max', 'accel_y_median', 'accel_y_skew', 'accel_y_kurtosis', 'accel_y_fft_energy', 'accel_y_fft_peak_amp', 'accel_y_fft_dominant_freq', 'accel_y_fft_centroid', 'accel_z_mean', 'accel_z_std', 'accel_z_var', 'accel_z_rms', 'accel_z_min', 'accel_z_max', 'accel_z_median', 'accel_z_skew', 'accel_z_kurtosis', 'accel_z_fft_energy', 'accel_z_fft_peak_amp', 'accel_z_fft_dominant_freq', 'accel_z_fft_centroid', 'gyro_z_mean', 'gyro_z_std', 'gyro_z_var', 'gyro_z_rms', 'gyro_z_min', 'gyro_z_max', 'gyro_z_median', 'gyro_z_skew', 'gyro_z_kurtosis', 'gyro_z_fft_energy', 'gyro_z_fft_peak_amp', 'gyro_z_fft_dominant_freq', 'gyro_z_fft_centroid', 'motor_current_1_mean', 'motor_curre

In [18]:
X_test_signal=signal_features_test.drop(columns=["flight_id","drone_id"])
print(X_test_signal.columns.tolist())

['accel_x_mean', 'accel_x_std', 'accel_x_var', 'accel_x_rms', 'accel_x_min', 'accel_x_max', 'accel_x_median', 'accel_x_skew', 'accel_x_kurtosis', 'accel_x_fft_energy', 'accel_x_fft_peak_amp', 'accel_x_fft_dominant_freq', 'accel_x_fft_centroid', 'accel_y_mean', 'accel_y_std', 'accel_y_var', 'accel_y_rms', 'accel_y_min', 'accel_y_max', 'accel_y_median', 'accel_y_skew', 'accel_y_kurtosis', 'accel_y_fft_energy', 'accel_y_fft_peak_amp', 'accel_y_fft_dominant_freq', 'accel_y_fft_centroid', 'accel_z_mean', 'accel_z_std', 'accel_z_var', 'accel_z_rms', 'accel_z_min', 'accel_z_max', 'accel_z_median', 'accel_z_skew', 'accel_z_kurtosis', 'accel_z_fft_energy', 'accel_z_fft_peak_amp', 'accel_z_fft_dominant_freq', 'accel_z_fft_centroid', 'gyro_z_mean', 'gyro_z_std', 'gyro_z_var', 'gyro_z_rms', 'gyro_z_min', 'gyro_z_max', 'gyro_z_median', 'gyro_z_skew', 'gyro_z_kurtosis', 'gyro_z_fft_energy', 'gyro_z_fft_peak_amp', 'gyro_z_fft_dominant_freq', 'gyro_z_fft_centroid', 'motor_current_1_mean', 'motor_curre

In [19]:
df_notes = pd.read_csv("data/train/train_notes.csv")
df_notes = df_notes.merge(
    df_tabular[["flight_id", "failure_within_horizon","drone_id"]],
    on="flight_id",
    how="left"
)

In [20]:
df_notes_test = pd.read_csv("data/test/test_notes.csv")
df_notes_test = df_notes_test.merge(
    df_tabular_test[["flight_id","drone_id"]],
    on="flight_id",
    how="left"
)

In [21]:
df_notes.info()

<class 'pandas.DataFrame'>
RangeIndex: 15872 entries, 0 to 15871
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   flight_id               15872 non-null  int64
 1   maintenance_note        11435 non-null  str  
 2   failure_within_horizon  15872 non-null  int64
 3   drone_id                15872 non-null  str  
dtypes: int64(2), str(2)
memory usage: 496.1 KB


In [22]:
df_notes_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 7534 entries, 0 to 7533
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   flight_id         7534 non-null   int64
 1   maintenance_note  5431 non-null   str  
 2   drone_id          7534 non-null   str  
dtypes: int64(1), str(2)
memory usage: 176.7 KB


In [23]:
df_notes["has_note"] = df_notes["maintenance_note"].notna().astype(int)
df_notes["note_length"] = df_notes["maintenance_note"].fillna("").str.split().str.len()
df_notes["maintenance_note"] = df_notes["maintenance_note"].fillna("")

In [24]:
df_notes_test["has_note"] = df_notes_test["maintenance_note"].notna().astype(int)
df_notes_test["note_length"] = df_notes_test["maintenance_note"].fillna("").str.split().str.len()
df_notes_test["maintenance_note"] = df_notes_test["maintenance_note"].fillna("")

In [25]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2), 
    max_features=60, 
    min_df=2,
     # FIX 1: Automatically drops generic English filler words
    stop_words='english', 
    # FIX 2: Ignores words that appear in more than 80% of your notes
    max_df=0.80 
    )
tfidf_matrix = tfidf.fit_transform(df_notes["maintenance_note"])
tfidf_matrix_test = tfidf.transform(df_notes_test["maintenance_note"])

In [26]:
feature_names = ["note_" + w.replace(" ", "_") for w in tfidf.get_feature_names_out()]

df_notes_features = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=feature_names
)

df_notes_features["has_note"] = df_notes["has_note"].values
df_notes_features["note_length"] = df_notes["note_length"].values
df_notes_features["flight_id"] = df_notes["flight_id"].values
df_notes_features["failure_within_horizon"] = df_notes["failure_within_horizon"].values
df_notes_features["drone_id"] = (
    df_tabular.set_index("flight_id")["drone_id"]
    .loc[df_notes["flight_id"]]
    .values
)
df_notes_features.head()

,note_anomalies,note_anomalies_observed,note_battery,note_battery_swapped,note_check,note_check_nominal,note_chip,note_cleaned,note_cleared,note_complete,...,note_tightened_landing,note_unrelated,note_unrelated_powertrain,note_updated,note_updated_flight,has_note,note_length,flight_id,failure_within_horizon,drone_id
0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.344839,0.344839,1,4,0,0,D1000
1,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,1.0,0.000000,0.0,...,0.0,0.0,0.0,0.000000,0.000000,1,5,1,0,D1000
2,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,1.0,0.000000,0.0,...,0.0,0.0,0.0,0.000000,0.000000,1,5,2,0,D1000
3,0.000000,0.000000,0.0,0.0,0.0,0.0,0.333333,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.000000,0.000000,1,6,3,0,D1000
4,0.377964,0.377964,0.0,0.0,0.0,0.0,0.000000,0.0,0.377964,0.0,...,0.0,0.0,0.0,0.000000,0.000000,1,6,4,0,D1000


In [27]:
feature_names_test = ["note_" + w.replace(" ", "_") for w in tfidf.get_feature_names_out()]

df_notes_features_test = pd.DataFrame(
    tfidf_matrix_test.toarray(),
    columns=feature_names_test
)

df_notes_features_test["has_note"] = df_notes_test["has_note"].values
df_notes_features_test["note_length"] = df_notes_test["note_length"].values
df_notes_features_test["flight_id"] = df_notes_test["flight_id"].values
df_notes_features_test["drone_id"] = (
    df_tabular_test.set_index("flight_id")["drone_id"]
    .loc[df_notes_test["flight_id"]]
    .values
)
df_notes_features_test.head()

,note_anomalies,note_anomalies_observed,note_battery,note_battery_swapped,note_check,note_check_nominal,note_chip,note_cleaned,note_cleared,note_complete,...,note_tightened,note_tightened_landing,note_unrelated,note_unrelated_powertrain,note_updated,note_updated_flight,has_note,note_length,flight_id,drone_id
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,1,7,45,D1002
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0,0,46,D1002
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,1,5,47,D1002
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.344839,0.344839,1,4,48,D1002
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0,0,49,D1002


In [28]:
df_notes.info()

<class 'pandas.DataFrame'>
RangeIndex: 15872 entries, 0 to 15871
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   flight_id               15872 non-null  int64
 1   maintenance_note        15872 non-null  str  
 2   failure_within_horizon  15872 non-null  int64
 3   drone_id                15872 non-null  str  
 4   has_note                15872 non-null  int64
 5   note_length             15872 non-null  int64
dtypes: int64(4), str(2)
memory usage: 744.1 KB


In [29]:
df_notes_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 7534 entries, 0 to 7533
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   flight_id         7534 non-null   int64
 1   maintenance_note  7534 non-null   str  
 2   drone_id          7534 non-null   str  
 3   has_note          7534 non-null   int64
 4   note_length       7534 non-null   int64
dtypes: int64(3), str(2)
memory usage: 294.4 KB


In [30]:
X_train_notes = df_notes_features.drop(columns=["flight_id", "failure_within_horizon", "drone_id"])


In [31]:
X_test_notes = df_notes_features_test.drop(columns=["flight_id", "drone_id"])


In [32]:
print(X_train_notes.shape)
print(X_train_signal.shape)
print(X_encoded_tabular.shape)
print(X_encoded_tabular.dtypes)
print(X_test_notes.shape)
print(X_test_signal.shape)
print(X_encoded_tabular_test.shape)
print(X_encoded_tabular_test.dtypes)

(15872, 62)
(15872, 104)
(15872, 29)
battery_capacity_mAh          int64
max_payload_g               float64
propeller_in                float64
manufacture_batch             int64
flight_index                  int64
payload_g                   float64
ambient_temp_C              float64
wind_speed_ms               float64
flight_duration_min         float64
avg_throttle                float64
num_aggressive_maneuvers      int64
cumulative_flight_hours     float64
battery_cycles                int64
payload_ratio               float64
throttle_x_wind             float64
maneuvers_per_min           float64
cycles_per_hour             float64
battery_cycles_sq             int64
motor_type_M1               float64
motor_type_M2               float64
motor_type_M3               float64
firmware_version_v3.1       float64
firmware_version_v3.2       float64
firmware_version_v4.0       float64
operator_region_east        float64
operator_region_north       float64
operator_region_south      

In [33]:
print(X_train_notes.index.equals(X_encoded_tabular.index))
print(X_train_notes.index.equals(X_encoded_tabular.index))
print(X_test_notes.index.equals(X_encoded_tabular_test.index))
print(X_test_notes.index.equals(X_encoded_tabular_test.index))

True
True
True
True


In [34]:
import pandas as pd

X_fusion = pd.concat(
    [X_encoded_tabular,
     X_train_signal,
     X_train_notes],
    axis=1
)

print(X_fusion.shape)


(15872, 195)


In [35]:
import pandas as pd

X_fusion_test = pd.concat(
    [X_encoded_tabular_test,
     X_test_signal,
     X_test_notes],
    axis=1
)

print(X_fusion_test.shape)


(7534, 195)


TESTING_LOGISTIC

In [36]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(penalty='l2',max_iter=1000))
])

In [37]:
# from sklearn.model_selection import GroupKFold, cross_val_score

# groups = df_tabular["drone_id"]

# cv = GroupKFold(n_splits=6)

# scores = cross_val_score(
#     pipe,
#     X_fusion,
#     labels,
#     cv=cv,
#     groups=groups,
#     scoring="average_precision"
# )

# print(scores)
# print(scores.mean())
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score

groups = df_tabular["drone_id"]

cv = StratifiedGroupKFold(
    n_splits=6,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    pipe,
    X_fusion,
    labels,
    cv=cv,
    groups=groups,
    scoring="average_precision"
)

print(scores)
print(scores.mean())

[0.49970654 0.57874561 0.49007433 0.45538902 0.48285327 0.52294531]
0.5049523465628466


TESTING- random forest 

In [38]:
from sklearn.ensemble import RandomForestClassifier

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier())
])

In [39]:
# groups=df_tabular["drone_id"]
# cv=GroupKFold(n_splits=6)
# scores=cross_val_score(
#     pipe,
#     X_fusion,
#     labels,
#     groups=groups,
#     scoring="average_precision"
# )
# print(scores)
# print(scores.mean())
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score

groups = df_tabular["drone_id"]

cv = StratifiedGroupKFold(
    n_splits=6,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    pipe,
    X_fusion,
    labels,
    cv=cv,
    groups=groups,
    scoring="average_precision"
)

print(scores)
print(scores.mean())

[0.43279845 0.55133124 0.46502898 0.38871938 0.43974472 0.46965689]
0.45787994537257043


TESTING_XGBOOST

In [40]:
from xgboost import XGBClassifier
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", XGBClassifier())
])




In [41]:
# groups=df_tabular["drone_id"]
# cv=GroupKFold(n_splits=6)
# scores=cross_val_score(
#     pipe,
#     X_fusion,
#     labels,
#     groups=groups,
#     scoring="average_precision"
# )
# print(scores)
# print(scores.mean())
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score

groups = df_tabular["drone_id"]

cv = StratifiedGroupKFold(
    n_splits=6,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    pipe,
    X_fusion,
    labels,
    cv=cv,
    groups=groups,
    scoring="average_precision"
)

print(scores)
print(scores.mean())

[0.44940426 0.52847132 0.48126128 0.4260358  0.43275111 0.42898996]
0.45781895513701865


TESTING DECISION TREES

In [42]:
from sklearn.tree import DecisionTreeClassifier
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", DecisionTreeClassifier())
])

In [43]:
# groups=df_tabular["drone_id"]
# cv=GroupKFold(n_splits=6)
# scores=cross_val_score(
#     pipe,
#     X_fusion,
#     labels,
#     groups=groups,
#     scoring="average_precision"
# )
# print(scores)
# print(scores.mean())
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score

groups = df_tabular["drone_id"]

cv = StratifiedGroupKFold(
    n_splits=6,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    pipe,
    X_fusion,
    labels,
    cv=cv,
    groups=groups,
    scoring="average_precision"
)

print(scores)
print(scores.mean())

[0.18645238 0.24038278 0.18319757 0.15790805 0.16112258 0.19105363]
0.18668616480800238


HYPERPARAMETER_TUNING Logistic 

In [44]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold, RandomizedSearchCV

cv = StratifiedGroupKFold(
    n_splits=6,
    shuffle=True,
    random_state=42
)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        penalty="l2",
        max_iter=3000
    ))
])

param_dist = {
    "clf__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "clf__class_weight": [None, "balanced"]
}

search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=12,
    cv=cv,
    scoring="average_precision",
    random_state=42,
    n_jobs=-1
)

search.fit(
    X_fusion,
    labels,
    groups=df_tabular["drone_id"]
)

print(search.best_score_)
print(search.best_params_)

0.5064280649013456
{'clf__class_weight': None, 'clf__C': 0.1}


In [45]:
print(X_fusion.columns.equals(X_fusion_test.columns))
print(list(X_fusion.columns))
print(list(X_fusion_test.columns))

True
['battery_capacity_mAh', 'max_payload_g', 'propeller_in', 'manufacture_batch', 'flight_index', 'payload_g', 'ambient_temp_C', 'wind_speed_ms', 'flight_duration_min', 'avg_throttle', 'num_aggressive_maneuvers', 'cumulative_flight_hours', 'battery_cycles', 'payload_ratio', 'throttle_x_wind', 'maneuvers_per_min', 'cycles_per_hour', 'battery_cycles_sq', 'motor_type_M1', 'motor_type_M2', 'motor_type_M3', 'firmware_version_v3.1', 'firmware_version_v3.2', 'firmware_version_v4.0', 'operator_region_east', 'operator_region_north', 'operator_region_south', 'operator_region_west', 'model_enc', 'accel_x_mean', 'accel_x_std', 'accel_x_var', 'accel_x_rms', 'accel_x_min', 'accel_x_max', 'accel_x_median', 'accel_x_skew', 'accel_x_kurtosis', 'accel_x_fft_energy', 'accel_x_fft_peak_amp', 'accel_x_fft_dominant_freq', 'accel_x_fft_centroid', 'accel_y_mean', 'accel_y_std', 'accel_y_var', 'accel_y_rms', 'accel_y_min', 'accel_y_max', 'accel_y_median', 'accel_y_skew', 'accel_y_kurtosis', 'accel_y_fft_ener

In [46]:
best_model = search.best_estimator_

pred = best_model.predict_proba(X_fusion_test)[:, 1]

submission = pd.DataFrame({
    "flight_id": df_tabular_test["flight_id"],
    "failure_within_horizon": pred
})

submission.to_csv("submission_logistic.csv", index=False)

HYPERPARAMETER_TUNING Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, RandomizedSearchCV

cv = StratifiedGroupKFold(
    n_splits=6,
    shuffle=True,
    random_state=42
)

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

param_dist = {
    "n_estimators": [200, 400, 600, 800],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"],
    "class_weight": [None, "balanced", "balanced_subsample"]
}

search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=30,
    cv=cv,
    scoring="average_precision",
    random_state=42,
    n_jobs=-1
)

search.fit(
    X_fusion,
    labels,
    groups=df_tabular["drone_id"]
)

print(search.best_score_)
print(search.best_params_)

In [ ]:
best_model = search.best_estimator_

pred = best_model.predict_proba(X_fusion_test)[:, 1]

submission = pd.DataFrame({
    "flight_id": df_tabular_test["flight_id"],
    "failure_within_horizon": pred
})

submission.to_csv("submission_rf.csv", index=False)

HYPERPARAMETER_TUNING XGBoost

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedGroupKFold, RandomizedSearchCV

cv = StratifiedGroupKFold(
    n_splits=6,
    shuffle=True,
    random_state=42
)

xgb = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

param_dist = {
    "n_estimators": [200, 400, 600, 800],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [3, 5, 7, 9],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.3],
    "scale_pos_weight": [1, 3, 5, 7]
}

search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=20,
    cv=cv,
    scoring="average_precision",
    random_state=42,
    n_jobs=-1
)

search.fit(
    X_fusion,
    labels,
    groups=df_tabular["drone_id"]
)

print(search.best_score_)
print(search.best_params_)

0.49958956452360054
{'subsample': 0.7, 'n_estimators': 400, 'min_child_weight': 1, 'max_depth': 5, 'learning_rate': 0.01, 'gamma': 0.3, 'colsample_bytree': 0.8}


In [ ]:
best_model = search.best_estimator_

pred = best_model.predict_proba(X_fusion_test)[:, 1]

submission = pd.DataFrame({
    "flight_id": df_tabular_test["flight_id"],
    "failure_within_horizon": pred
})

submission.to_csv("submission_xgb.csv", index=False)

HYPERPARMETER TUNING FOR DECISION TREES

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedGroupKFold, RandomizedSearchCV

# 1. Define the cross-validation strategy
cv = StratifiedGroupKFold(
    n_splits=6,
    shuffle=True,
    random_state=42
)

# 2. Initialize the Decision Tree Classifier
dt = DecisionTreeClassifier(random_state=42)

# 3. Define the hyperparameter distribution
param_dist_dt = {
    "criterion": ["gini", "entropy", "log_loss"],
    "max_depth": [None, 5, 10, 15, 20, 30],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": [None, "sqrt", "log2"],
    "class_weight": [None, "balanced"]
}

# 4. Set up the Randomized Search
search_dt = RandomizedSearchCV(
    estimator=dt,
    param_distributions=param_dist_dt,
    n_iter=20,
    cv=cv,
    scoring="average_precision",
    random_state=42,
    n_jobs=-1
)

# 5. Fit the model
search_dt.fit(
    X_fusion,
    labels,
    groups=df_tabular["drone_id"]
)

# 6. Print the results
print("Best PR-AUC Score:", search_dt.best_score_)
print("Best Decision Tree Params:", search_dt.best_params_)